In [4]:
import json
from kafka import KafkaConsumer

In [5]:
topic_name = "app-events-piyush"
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers="localhost:9092",
    auto_offset_reset="latest",
    enable_auto_commit=False,
    group_id="app-events-piyush-demo",
    consumer_timeout_ms=60000
)

while not consumer.assignment():
    consumer.poll(timeout_ms=1000)
print("Consumer is ready. Run the producer now.")

Consumer is ready. Run the producer now.


In [6]:
number_of_messages = 11
event_counts = {"LOGIN": 0, "SEARCH": 0, "PURCHASE": 0}
invalid_count = 0
processed_count = 0

print("Offset = the record's position in its Kafka partition.")

try:
    for _ in range(number_of_messages):
        message = next(consumer)
        processed_count += 1
        print(f"\nPartition: {message.partition}, Offset: {message.offset}")

        try:
            event = json.loads(message.value.decode("utf-8"))
        except json.JSONDecodeError:
            invalid_count += 1
            print("Invalid message: the message is not valid JSON.")
            continue

        if not isinstance(event, dict):
            invalid_count += 1
            print("Invalid message: expected a JSON object.")
            continue

        if not event.get("event_id") or not event.get("event_type"):
            invalid_count += 1
            print("Invalid message: event_id or event_type is missing.")
            continue

        event_type = str(event["event_type"]).upper()
        device = str(event.get("device") or "UNKNOWN").upper()

        if event_type in event_counts:
            event_counts[event_type] += 1
            print(
                "Event:", event["event_id"],
                "User:", event.get("user_id", "UNKNOWN"),
                "Type:", event_type,
                "Device:", device
            )
        else:
            invalid_count += 1
            print("Invalid message: event_type must be LOGIN, SEARCH, or PURCHASE.")
except StopIteration:
    print("No more messages arrived within 60 seconds.")
finally:
    consumer.close()

print("\n--- FINAL SUMMARY ---")
print("Messages processed:", processed_count)
print("Invalid messages:", invalid_count)
print("Event counts:", event_counts)

Offset = the record's position in its Kafka partition.

Partition: 0, Offset: 56
Event: E101 User: U101 Type: LOGIN Device: MOBILE

Partition: 0, Offset: 57
Event: E102 User: U102 Type: PURCHASE Device: WEB

Partition: 0, Offset: 58
Event: E103 User: U103 Type: SEARCH Device: MOBILE

Partition: 0, Offset: 59
Event: E104 User: U104 Type: LOGIN Device: WEB

Partition: 0, Offset: 60
Event: E105 User: U105 Type: SEARCH Device: MOBILE

Partition: 0, Offset: 61
Event: E106 User: U106 Type: PURCHASE Device: WEB

Partition: 0, Offset: 62
Event: E107 User: U107 Type: LOGIN Device: MOBILE

Partition: 0, Offset: 63
Event: E108 User: U108 Type: SEARCH Device: WEB

Partition: 0, Offset: 64
Event: E109 User: U109 Type: PURCHASE Device: MOBILE

Partition: 0, Offset: 65
Event: E110 User: U110 Type: LOGIN Device: WEB

Partition: 0, Offset: 66
Invalid message: event_id or event_type is missing.

--- FINAL SUMMARY ---
Messages processed: 11
Invalid messages: 1
Event counts: {'LOGIN': 4, 'SEARCH': 3, 'PUR